In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import hashlib
import json
import os
import warnings
import shutil

RESULT_DIR = RESULTS
RIDGE_PRED_DIR = RESULT_DIR / 'ridge_baseline_alpha_extended'
RIDGE_TRUTH_DIR = RESULT_DIR / 'ridge_baseline'
RIDGE_ALPHA = 1e-8
FIGURE_DIR = FIGURES
WORK_DIR = RESULTS / 'fig7_s13_alpha1e8'
FIGURE_STEM_OVERRIDES = {'Fig7': None, 'FigS13': None}
EXPECTED_COMMON_COUNT = 42064
FIG_DPI = 800
SHOW_FIGURES = True
REUSE_OWN_CACHE = True
BLOCK_SIZE = 32
FONT_SCALE = 1.0

FILE_OVERRIDES = {}
VARIABLE_OVERRIDES = {}

W_TO_OMEGA = {'B': -10.0, 'P': -10.0, 'Ridge': 1.0}
LEADS = tuple(range(1, 7))
COMPONENTS = ('u', 'v', 'w')
GROUPS = ('B', 'P', 'Ridge')
MODEL_LABELS = {'B': 'DA-TransUNet', 'P': 'Persistence', 'Ridge': 'Ridge'}
KINDS = ('testy', 'predicty')
METRIC_NAMES = ('R', 'RMSPE', 'RMSE', 'MAE')
TIME_ROUND_TOL_NS = 1000
SCHEMA_VERSION = 'fig7-s13-ridge-alpha1e8-v1.0'
EXPECTED_NATIVE_COUNT = {1: 42071, 2: 42071, 3: 42070, 4: 42070, 5: 42070, 6: 42070}

BASIN_MASK_PATH = None
BASIN_MASK_VARIABLE = 'basin_mask'
BOUNDARY_PATH = None

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(WORK_DIR / 'matplotlib_config'))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
from IPython.display import display

LEVELS = np.array([850., 925., 1000.])
CONFIGS = [(g, h) for g in GROUPS for h in LEADS]
ALIASES = {
    'time': ('time', 'valid_time', 'times', 'date', 'datetime'),
    'level': ('level', 'levels', 'pressure_level', 'isobaricInhPa', 'isobaricInPa', 'plev'),
    'latitude': ('latitude', 'lat'),
    'longitude': ('longitude', 'lon'),
}
if not isinstance(BLOCK_SIZE, int) or BLOCK_SIZE < 1:
    raise ValueError('BLOCK_SIZE必须为正整数。')
if FIG_DPI <= 600:
    raise ValueError('FIG_DPI必须大于600。')
for group, factor in W_TO_OMEGA.items():
    if not np.isfinite(factor) or factor == 0:
        raise ValueError(f'{group}: 垂直速度转换系数必须为非零有限值。')
if W_TO_OMEGA['P'] != W_TO_OMEGA['B']:
    raise ValueError('Persistence复用B的testy，二者单位转换必须相同。')


In [ ]:
def rounded_hour_times(values, source):
    raw = np.asarray(values)
    if not np.issubdtype(raw.dtype, np.datetime64):
        raise TypeError(f'{source}: time须可解码为datetime64，不能将非标准日历静默转为标准日历。')
    raw = raw.astype('datetime64[ns]')
    if np.isnat(raw).any() or raw.size == 0:
        raise ValueError(f'{source}: 空时间轴或NaT。')
    ns = raw.astype(np.int64)
    if np.any(np.diff(ns) <= 0):
        raise ValueError(f'{source}: 原时间轴重复或倒序。')
    hour = 3_600_000_000_000
    quotient, remainder = np.divmod(ns, hour)
    rounded_ns = (quotient + (remainder >= hour // 2)) * hour
    delta = rounded_ns - ns
    if np.max(np.abs(delta)) > TIME_ROUND_TOL_NS:
        raise ValueError(f'{source}: 非整点偏差{np.max(np.abs(delta))} ns超过允许的编码噪声。')
    if np.any(np.diff(rounded_ns) <= 0):
        raise ValueError(f'{source}: 整点编码修正后出现重复或倒序。')
    changes = [dict(index=int(i), original=str(raw[i]), rounded=str(np.datetime64(int(rounded_ns[i]), 'ns')),
                    correction_ns=int(delta[i])) for i in np.flatnonzero(delta)]
    return rounded_ns.astype('datetime64[ns]'), changes


def canonical_field(ds, variable, source):
    if variable in ds.data_vars:
        da = ds[variable]
    else:
        candidates = []
        for name, value in ds.data_vars.items():
            if not np.issubdtype(value.dtype, np.number):
                continue
            if all(any(alias in value.dims for alias in aliases) for aliases in ALIASES.values()):
                candidates.append(name)
        if len(candidates) != 1:
            raise ValueError(f'{source}: 找不到{variable}且无法唯一确定四维风场，候选={candidates}。')
        da = ds[candidates[0]]
    rename = {}
    for standard, aliases in ALIASES.items():
        found = [a for a in aliases if a in da.dims]
        if len(found) != 1:
            raise ValueError(f'{source}: {standard}维度不存在或有歧义：{found}。')
        rename[found[0]] = standard
    da = da.rename({a: b for a, b in rename.items() if a != b})
    for dim in tuple(da.dims):
        if dim not in ALIASES:
            if da.sizes[dim] != 1:
                raise ValueError(f'{source}: 不能自动丢弃额外非单例维度{dim}。')
            da = da.isel({dim: 0}, drop=True)
    for dim in ALIASES:
        if dim not in da.coords or da[dim].ndim != 1:
            raise ValueError(f'{source}: {dim}必须有一维坐标。')
    lev = np.asarray(da.level.values, dtype=np.float64)
    units = str(da.level.attrs.get('units', '')).strip().lower().replace(' ', '')
    if units in ('pa', 'pascal', 'pascals') or (not units and np.all(lev > 2000)):
        lev = lev / 100.0
    elif units not in ('', 'hpa', 'hectopascal', 'hectopascals', 'mb', 'millibar', 'millibars'):
        raise ValueError(f'{source}: 未识别的气压层单位{units!r}。')
    if len(np.unique(lev)) != len(lev):
        raise ValueError(f'{source}: 气压层坐标重复。')
    indices = []
    for wanted in LEVELS:
        hit = np.flatnonzero(np.isclose(lev, wanted, rtol=0, atol=1e-8))
        if len(hit) != 1:
            raise ValueError(f'{source}: 缺少或重复气压层{wanted} hPa。')
        indices.append(int(hit[0]))
    da = da.assign_coords(level=lev).isel(level=indices)
    for dim in ('latitude', 'longitude'):
        c = np.asarray(da[dim].values, dtype=np.float64)
        if not np.isfinite(c).all() or len(np.unique(c)) != len(c):
            raise ValueError(f'{source}: {dim}坐标含非有限值或重复。')
        da = da.isel({dim: np.argsort(c)}).assign_coords({dim: np.sort(c)})
    da = da.transpose('time', 'level', 'latitude', 'longitude')
    return da


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def file_stat(path):
    stat = Path(path).stat()
    return dict(path=str(Path(path).resolve()), size=stat.st_size, mtime_ns=stat.st_mtime_ns)


def time_selection(indices):
    if len(indices) and (len(indices) == 1 or np.all(np.diff(indices) == 1)):
        return slice(int(indices[0]), int(indices[-1]) + 1)
    return indices


def nc_path(group, lead, component, kind):
    if group == 'P':
        return nc_path('B', lead, component, 'testy')
    key = (group, int(lead), component, kind)
    default = ((RIDGE_PRED_DIR if kind == 'predicty' else RIDGE_TRUTH_DIR) / f'Ridge_{lead}hour_{kind}.nc' if group == 'Ridge' else
               RESULT_DIR / f'da_transunet_{kind}_{component}_{lead}_24year_changjiang_residual.nc')
    return Path(FILE_OVERRIDES.get(key, default))


def variable_name(ds, group, lead, component, kind, path):
    source_group = 'B' if group == 'P' else group
    source_kind = 'testy' if group == 'P' else kind
    key = (source_group, int(lead), component, source_kind)
    if key in VARIABLE_OVERRIDES:
        name = VARIABLE_OVERRIDES[key]
        if name not in ds.data_vars:
            raise ValueError(f'{path}: 指定变量{name!r}不存在，实际为{list(ds.data_vars)}。')
        return name
    candidates = [f'{source_kind}_{component}', component]
    if component == 'w':
        candidates += [f'{source_kind}_omega', 'omega']
    for name in candidates:
        if name in ds.data_vars:
            return name
    hits = [name for name, da in ds.data_vars.items()
            if np.issubdtype(da.dtype, np.number)
            and all(any(a in da.dims for a in aliases) for aliases in ALIASES.values())]
    if len(hits) == 1 and source_group == 'B':
        return hits[0]
    raise ValueError(f'{path}: 无法识别{source_kind}/{component}，实际变量={list(ds.data_vars)}；'
                     '请在VARIABLE_OVERRIDES中填写精确映射。')


def scan_inputs():
    records, audit = {}, []
    common, latitude, longitude = None, None, None
    missing = sorted({str(nc_path(g, h, c, k)) for g, h in CONFIGS
                      for c in COMPONENTS for k in KINDS
                      if not nc_path(g, h, c, k).is_file()})
    if missing:
        raise FileNotFoundError('缺少以下文件，请检查目录或FILE_OVERRIDES：\n' + '\n'.join(missing))
    for group, lead in CONFIGS:
        for component in COMPONENTS:
            for kind in KINDS:
                key = (group, lead, component, kind)
                path = nc_path(*key)
                with xr.open_dataset(path, cache=False) as source:
                    variable = variable_name(source, *key, path)
                    field = canonical_field(source, variable, path)
                    native, corrections = rounded_hour_times(field.time.values, path)
                    lat, lon = field.latitude.values.copy(), field.longitude.values.copy()
                    if latitude is None:
                        latitude, longitude = lat, lon
                    elif not (np.array_equal(latitude, lat) and np.array_equal(longitude, lon)):
                        raise ValueError(f'{path}: 网格不一致，不进行空间插值。')
                    source_units = str(field.attrs.get('units', 'not specified'))
                target = native + np.timedelta64(lead, 'h') if group == 'P' and kind == 'predicty' else native
                common = target if common is None else np.intersect1d(common, target, assume_unique=True)
                records[key] = dict(path=path, variable=variable, native_times=native, times=target, stat=file_stat(path))
                audit.append(dict(group=group, lead_hour=lead, component=component, kind=kind,
                                  variable=variable, count=len(native), native_start=str(native[0]),
                                  native_end=str(native[-1]), stored_units=source_units,
                                  conversion_to_physical=W_TO_OMEGA[group] if component == 'w' else 1.,
                                  n_time_round_corrections=len(corrections), **file_stat(path)))
    for h in LEADS:
        baseline = records[('B', h, 'u', 'testy')]['native_times']
        if EXPECTED_NATIVE_COUNT is not None and len(baseline) != EXPECTED_NATIVE_COUNT[h]:
            raise ValueError(f'{h}h原生样本数{len(baseline)}与验收记录{EXPECTED_NATIVE_COUNT[h]}不符。请核对文件。')
        for group in ('B', 'Ridge'):
            for c in COMPONENTS:
                for kind in KINDS:
                    if not np.array_equal(records[(group, h, c, kind)]['native_times'], baseline):
                        raise ValueError(f'{group}/{h}h/{c}/{kind}: 原生时间轴与该时效Fig7基准不一致。')
    if common is None or len(common) < 3:
        raise ValueError('共同评价时次不足3个。')
    if np.any(np.diff(common) != np.timedelta64(1, 'h')):
        raise ValueError('共同评价期存在非逐小时间断，请核对文件；不自动填补。')
    for entry, record in zip(audit, records.values()):
        idx = np.searchsorted(record['times'], common)
        if not np.array_equal(record['times'][idx], common):
            raise AssertionError('时间戳索引失败。')
        record['indices'] = idx
        excluded = np.setdiff1d(record['times'], common)
        entry.update(evaluation_count=len(common), excluded_count=len(excluded),
                     excluded_first=str(excluded[0]) if len(excluded) else None,
                     excluded_last=str(excluded[-1]) if len(excluded) else None)
    return records, common, latitude, longitude, audit


def make_spatial_mask():
    mask = np.ones((len(latitude), len(longitude)), dtype=bool)
    if BOUNDARY_PATH is not None:
        import geopandas as gpd
        import shapely
        boundary = gpd.read_file(BOUNDARY_PATH)
        if boundary.crs is None:
            raise ValueError('流域边界缺少CRS。')
        boundary = boundary.to_crs(4326)
        geom = boundary.geometry.union_all() if hasattr(boundary.geometry, 'union_all') else boundary.geometry.unary_union
        xx, yy = np.meshgrid(longitude, latitude)
        mask &= np.asarray(shapely.covers(geom, shapely.points(xx, yy)), dtype=bool)
    if BASIN_MASK_PATH is not None:
        with xr.open_dataset(BASIN_MASK_PATH) as source:
            da = source[BASIN_MASK_VARIABLE]
            ren = {a: b for b in ('latitude', 'longitude') for a in ALIASES[b] if a in da.dims and a != b}
            da = da.rename(ren).transpose('latitude', 'longitude').sortby('latitude').sortby('longitude')
            if not np.array_equal(da.latitude.values, latitude) or not np.array_equal(da.longitude.values, longitude):
                raise ValueError('流域掩膜与风场网格不一致。')
            mask &= np.isfinite(da.values) & (da.values > 0)
    if not mask.any():
        raise ValueError('空间掩膜没有有效格点。')
    return mask


def assert_sources_unchanged():
    for record in records.values():
        if file_stat(record['path']) != record['stat']:
            raise RuntimeError(f"源文件发生变化：{record['path']}。请重新运行输入核对框。")


def exact_truth_check(a, b, label):
    if a.shape != b.shape or not np.array_equal(a, b, equal_nan=True):
        good = np.isfinite(a) & np.isfinite(b) if a.shape == b.shape else np.array(False)
        delta = float(np.max(np.abs(a[good] - b[good]))) if good.any() else None
        raise ValueError(f'{label}: Ridge与原模型物理单位真值不完全一致，最大有限差={delta}。'
                         '请核对源文件和w转换；不自动放宽容差或替换真值。')


def atomic_npz(path, **payload):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **payload)
    os.replace(temporary, path)


def write_nc(ds, path):
    path = Path(path)
    temporary = path.with_name(path.stem + '.tmp.nc')
    ds.to_netcdf(temporary, engine='netcdf4',
                 encoding={k: {'zlib': True, 'complevel': 4} for k in ds.data_vars})
    with xr.open_dataset(temporary) as check:
        check.load()
        if dict(check.sizes) != dict(ds.sizes):
            raise RuntimeError('指标NC读回维度不一致。')
        for name in ds.data_vars:
            if not np.array_equal(check[name].values, ds[name].values, equal_nan=True):
                raise RuntimeError(f'指标NC读回不一致：{name}')
    os.replace(temporary, path)


def audit_ridge_headers():
    rows = []
    for h in LEADS:
        pred = nc_path('Ridge', h, 'u', 'predicty')
        truth = nc_path('Ridge', h, 'u', 'testy')
        if pred.resolve() == (RIDGE_TRUTH_DIR / f'Ridge_{h}hour_predicty.nc').resolve():
            raise ValueError(f'{h}h误用了旧alpha=1e-6预测：{pred}')
        with xr.open_dataset(pred, cache=False) as ds:
            found = {}
            for name in ('alpha', 'ridge_alpha', 'selected_alpha', 'regularization_alpha'):
                if name in ds.attrs:
                    value = float(ds.attrs[name])
                    if not np.isclose(value, RIDGE_ALPHA, rtol=1e-10, atol=0):
                        raise ValueError(f'{pred}: {name}={value}，不是要求的{RIDGE_ALPHA}。')
                    found[name] = value
            if 'downstream_w_scale' in ds.attrs and float(ds.attrs['downstream_w_scale']) != 1.:
                raise ValueError(f'{pred}: downstream_w_scale必须为1。')
        rows.append(dict(lead_hour=h, declared_alpha=RIDGE_ALPHA, predicty=str(pred), testy=str(truth),
                         alpha_fields_in_nc=found,
                         provenance='New prediction directory from the supplied final boundary report; '
                                    'numeric alpha metadata checked when present'))
    return rows


In [ ]:
records, common_times, latitude, longitude, source_audit = scan_inputs()
if EXPECTED_COMMON_COUNT is not None and len(common_times) != EXPECTED_COMMON_COUNT:
    raise ValueError(f'共同评价期为{len(common_times)}小时，与原图的{EXPECTED_COMMON_COUNT}不一致。请核对输入。')
ridge_source_audit = audit_ridge_headers()
basin_mask = make_spatial_mask()
settings = dict(
    schema=SCHEMA_VERSION, ridge_alpha=RIDGE_ALPHA, leads=list(LEADS), groups=list(GROUPS), components=list(COMPONENTS),
    levels=LEVELS.tolist(), w_to_omega=W_TO_OMEGA, time_round_tolerance_ns=TIME_ROUND_TOL_NS,
    time_sha256=hashlib.sha256(common_times.astype('<i8').tobytes()).hexdigest(),
    mask_sha256=hashlib.sha256(basin_mask.tobytes()).hexdigest(),
    definition='grid temporal metrics; RMSPE=RMSE/range; equal grid then equal three levels',
    common_finite_mask='all methods and leads per component/level/grid/time',
    inputs=[dict(key=list(key), variable=record['variable'], **record['stat']) for key, record in records.items()],
)
input_signature = hashlib.sha256(json.dumps(settings, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()


In [ ]:
def merge_moments(old_n, old_mean_x, old_mean_y, old_m2_x, old_m2_y, old_cov,
                  x, y, valid):
    bn = valid.sum(axis=0).astype(np.int64)
    bx = np.divide(np.where(valid, x, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    by = np.divide(np.where(valid, y, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    dx = np.where(valid, x - bx, 0.0)
    dy = np.where(valid, y - by, 0.0)
    bm2x = np.sum(dx * dx, axis=0)
    bm2y = np.sum(dy * dy, axis=0)
    bcov = np.sum(dx * dy, axis=0)
    total = old_n + bn
    fraction = np.divide(bn, total, out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    factor = np.divide(old_n.astype(np.float64) * bn, total,
                       out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    shift_x = bx - old_mean_x
    shift_y = by - old_mean_y
    old_mean_x += shift_x * fraction
    old_mean_y += shift_y * fraction
    old_m2_x += bm2x + shift_x * shift_x * factor
    old_m2_y += bm2y + shift_y * shift_y * factor
    old_cov += bcov + shift_x * shift_y * factor
    return bn


def compute_grid_metrics():
    assert_sources_unchanged()
    dims = ('group', 'lead_hour', 'component', 'level', 'latitude', 'longitude')
    shape = (len(GROUPS), len(LEADS), len(COMPONENTS), len(LEVELS), len(latitude), len(longitude))
    output = {name: np.full(shape, np.nan, dtype=np.float64) for name in METRIC_NAMES}
    ranges = np.full(shape, np.nan, dtype=np.float64)
    counts = np.zeros(shape[2:], dtype=np.int64)
    spatial_shape = shape[3:]
    cfg_shape = (len(CONFIGS),) + spatial_shape
    truth_audit = []
    for ci, component in enumerate(COMPONENTS):
        cache_path = WORK_DIR / f'component_{component}_metrics.npz'
        if REUSE_OWN_CACHE and cache_path.is_file():
            with np.load(cache_path, allow_pickle=False) as cached:
                if str(cached['input_signature'].item()) == input_signature:
                    expected = (len(GROUPS), len(LEADS)) + spatial_shape
                    if all(cached[name].shape == expected for name in METRIC_NAMES):
                        for name in METRIC_NAMES:
                            output[name][:, :, ci] = cached[name]
                        ranges[:, :, ci] = cached['reference_range']
                        counts[ci] = cached['n_valid']
                        truth_audit.extend(json.loads(str(cached['truth_audit'].item())))
                        print(f'{component}: 复用本notebook同一输入签名的已完成缓存。', flush=True)
                        continue
        n = np.zeros(spatial_shape, dtype=np.int64)
        mean_t, mean_p, m2_t, m2_p, cov, sse, sae = [np.zeros(cfg_shape, dtype=np.float64) for _ in range(7)]
        truth_min = np.full(cfg_shape, np.inf)
        truth_max = np.full(cfg_shape, -np.inf)
        component_truth_audit = []
        with ExitStack() as stack:
            datasets, fields = {}, {}
            for group, lead in CONFIGS:
                for kind in KINDS:
                    key = (group, lead, component, kind)
                    record = records[key]
                    path = record['path']
                    if path not in datasets:
                        datasets[path] = stack.enter_context(xr.open_dataset(path, cache=False))
                    field_key = (path, record['variable'])
                    if field_key not in fields:
                        fields[field_key] = canonical_field(datasets[path], record['variable'], path)

            def read_values(group, lead, kind, indices):
                record = records[(group, lead, component, kind)]
                field = fields[(record['path'], record['variable'])]
                values = np.asarray(field.isel(time=time_selection(indices)).values, dtype=np.float64)
                factor = W_TO_OMEGA[group] if component == 'w' else 1.
                return values if factor == 1. else values * factor

            for lead in LEADS:
                record = records[('B', lead, component, 'testy')]
                outside = np.setdiff1d(np.arange(len(record['native_times'])), record['indices'])
                for start in range(0, len(outside), BLOCK_SIZE):
                    idx = outside[start:start + BLOCK_SIZE]
                    exact_truth_check(read_values('B', lead, 'testy', idx),
                                      read_values('Ridge', lead, 'testy', idx),
                                      f'{component}/{lead}h/共同期以外')

            for start in range(0, len(common_times), BLOCK_SIZE):
                stop = min(start + BLOCK_SIZE, len(common_times))
                loaded = {}
                for lead in LEADS:
                    for group, kind in [('B', 'testy'), ('B', 'predicty'),
                                        ('P', 'predicty'), ('Ridge', 'testy'), ('Ridge', 'predicty')]:
                        key = (group, lead, component, kind)
                        loaded[(group, lead, kind)] = read_values(group, lead, kind, records[key]['indices'][start:stop])
                    exact_truth_check(loaded[('B', lead, 'testy')], loaded[('Ridge', lead, 'testy')],
                                      f'{component}/{lead}h/时次{start}:{stop}')
                pairs, valid = [], np.ones((stop - start,) + spatial_shape, dtype=bool)
                for group, lead in CONFIGS:
                    truth = loaded[('B' if group == 'P' else group, lead, 'testy')]
                    prediction = loaded[(group, lead, 'predicty')]
                    valid &= np.isfinite(truth) & np.isfinite(prediction)
                    pairs.append((truth, prediction))
                bn = valid.sum(axis=0)
                for k, (truth, prediction) in enumerate(pairs):
                    merge_moments(n, mean_t[k], mean_p[k], m2_t[k], m2_p[k], cov[k], truth, prediction, valid)
                    error = np.where(valid, prediction - truth, 0.)
                    sse[k] += np.sum(error * error, axis=0)
                    sae[k] += np.sum(np.abs(error), axis=0)
                    truth_min[k] = np.minimum(truth_min[k], np.min(np.where(valid, truth, np.inf), axis=0))
                    truth_max[k] = np.maximum(truth_max[k], np.max(np.where(valid, truth, -np.inf), axis=0))
                n += bn
                del loaded, pairs, truth, prediction, error, valid
                if start == 0 or stop == len(common_times) or stop % (BLOCK_SIZE * 100) == 0:
                    print(f'{component}: {stop:,}/{len(common_times):,} 个共同目标时次', flush=True)

        enough = n >= 3
        for k, (group, lead) in enumerate(CONFIGS):
            where = (GROUPS.index(group), LEADS.index(lead), ci)
            rmse = np.sqrt(np.divide(sse[k], n, out=np.full(spatial_shape, np.nan), where=enough))
            mae = np.divide(sae[k], n, out=np.full(spatial_shape, np.nan), where=enough)
            denominator = np.sqrt(m2_t[k]) * np.sqrt(m2_p[k])
            correlation = np.divide(cov[k], denominator, out=np.full(spatial_shape, np.nan),
                                    where=enough & (denominator > 0))
            finite = np.isfinite(correlation)
            if np.any(np.abs(correlation[finite]) > 1. + 1e-10):
                raise ArithmeticError('相关系数超出数值舍入允许范围。')
            correlation[finite] = np.clip(correlation[finite], -1., 1.)
            with np.errstate(invalid='ignore'):
                truth_range = truth_max[k] - truth_min[k]
            truth_range = np.where(enough & np.isfinite(truth_range), truth_range, np.nan)
            rmspe = np.divide(rmse, truth_range, out=np.full(spatial_shape, np.nan),
                              where=enough & np.isfinite(truth_range) & (truth_range > 0))
            for name, value in [('R', correlation), ('RMSPE', rmspe), ('RMSE', rmse), ('MAE', mae)]:
                output[name][where] = value
            ranges[where] = truth_range
        counts[ci] = n
        assert_sources_unchanged()
        for lead in LEADS:
            component_truth_audit.append(dict(component=component, lead_hour=lead,
                native_count=len(records[('B', lead, component, 'testy')]['native_times']),
                full_native_truth_equal=True, comparison='float64 physical units; exact equality including NaN mask'))
        truth_audit.extend(component_truth_audit)
        atomic_npz(cache_path, input_signature=np.array(input_signature),
                   truth_audit=np.array(json.dumps(component_truth_audit)),
                   n_valid=n, reference_range=ranges[:, :, ci],
                   **{name: output[name][:, :, ci] for name in METRIC_NAMES})
        del mean_t, mean_p, m2_t, m2_p, cov, sse, sae, truth_min, truth_max

    coords = dict(group=list(GROUPS), lead_hour=list(LEADS), component=list(COMPONENTS),
                  level=LEVELS, latitude=latitude, longitude=longitude)
    ds = xr.Dataset({name: (dims, value) for name, value in output.items()}, coords=coords)
    ds['reference_range'] = (dims, ranges)
    ds['n_valid'] = (dims[2:], counts)
    ds['basin_mask'] = (('latitude', 'longitude'), basin_mask.astype(np.uint8))
    for name in METRIC_NAMES:
        mask = np.all(np.isfinite(output[name]), axis=(0, 1))
        ds[f'common_spatial_mask_{name}'] = (dims[2:], mask.astype(np.uint8))
    ds = ds.assign_coords(evaluation_time=('evaluation_time', common_times))
    ds.level.attrs['units'] = 'hPa'
    ds.R.attrs.update(units='1', definition='Pearson r over common finite evaluation times')
    ds.RMSPE.attrs.update(units='1', definition='RMSE/(max(truth)-min(truth)); no multiplication by 100')
    for name in ('RMSE', 'MAE', 'reference_range'):
        ds[name].attrs['component_units'] = 'u,v: m s-1; w denotes omega: Pa s-1'
    ds.attrs.update(schema_version=SCHEMA_VERSION, input_signature=input_signature,
                    ridge_alpha=RIDGE_ALPHA, ridge_prediction_directory=str(RIDGE_PRED_DIR),
                    common_count=len(common_times), common_start=str(common_times[0]), common_end=str(common_times[-1]),
                    persistence='Own-lead reference at target time minus lead, matched by timestamp',
                    aggregation='Temporal grid metrics; equal grid means within each level; equal three-level means',
                    finite_mask='All three methods and six leads, separately per component/level/grid/time',
                    w_conversion_json=json.dumps(W_TO_OMEGA),
                    full_native_truth_checks=json.dumps(truth_audit),
                    created_utc=datetime.now(timezone.utc).isoformat())
    assert_sources_unchanged()
    return ds


def lead_summaries(ds):
    rows = []
    for group in GROUPS:
        for h in LEADS:
            for component in COMPONENTS:
                for level in LEVELS:
                    row = dict(model=MODEL_LABELS[group], group=group, lead_hour=h,
                               component=component, level_hpa=int(level))
                    for metric in METRIC_NAMES:
                        mask = basin_mask & ds[f'common_spatial_mask_{metric}'].sel(component=component, level=level).values.astype(bool)
                        values = ds[metric].sel(group=group, lead_hour=h, component=component, level=level).values
                        row[metric] = float(values[mask].mean()) if mask.any() else np.nan
                        row[metric + '_n_grid'] = int(mask.sum())
                    rows.append(row)
    return pd.DataFrame(rows)


def make_curve_table(summary):
    rows = []
    for group in GROUPS:
        for lead in LEADS:
            for component in COMPONENTS:
                selected = summary.loc[(summary.group == group) & (summary.lead_hour == lead)
                                       & (summary.component == component)]
                row = dict(model=MODEL_LABELS[group], group=group, lead_hour=lead, component=component)
                for metric in METRIC_NAMES:
                    values = selected[metric].to_numpy()
                    row[metric] = float(values.mean()) if len(values) == 3 and np.isfinite(values).all() else np.nan
                rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
metrics = compute_grid_metrics()
summary = lead_summaries(metrics)
curves = make_curve_table(summary)
write_nc(metrics, WORK_DIR / 'Fig7_S13_alpha1e8_grid_metrics.nc')
summary.to_csv(WORK_DIR / 'Fig7_S13_alpha1e8_metrics_by_level.csv', index=False, encoding='utf-8-sig')
curves.to_csv(WORK_DIR / 'Fig7_S13_alpha1e8_curve_values.csv', index=False, encoding='utf-8-sig')
